<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Predicción del aterrizaje de la primera etapa del Falcon 9 de SpaceX**


 ## Laboratorio 2: Procesamiento de datos


Tiempo Estimado Necesario: **60** minutos


En este laboratorio, realizaremos un análisis exploratorio de datos (AED) para encontrar patrones y determinar las etiquetas para entrenar modelos supervisados.

En el conjunto de datos, existen varios casos en los que el propulsor no aterrizó correctamente. En ocasiones, se intentó un aterrizaje, pero falló debido a un accidente; por ejemplo, "True Ocean" significa que la misión aterrizó con éxito en una región específica del océano, mientras que "False Ocean" significa que no se logró aterrizar en dicha región. "True RTLS" significa que la misión aterrizó con éxito en una plataforma terrestre, mientras que "False RTLS" significa que no se logró aterrizar en una plataforma terrestre. "True ASDS" significa que la misión aterrizó con éxito en una plataforma flotante no tripulada, mientras que "False ASDS" significa que no se logró aterrizar en una plataforma flotante no tripulada.

En este laboratorio convertiremos principalmente esos resultados en etiquetas de entrenamiento donde `1` significa que el propulsor aterrizó con éxito y `0` significa que no tuvo éxito.


La primera etapa del Falcon 9 aterrizará con éxito.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


## Objetivos
Realizar análisis exploratorio de datos y determinar etiquetas de entrenamiento

- Análisis exploratorio de datos
- Determinar etiquetas de entrenamiento

----


Instale las siguientes bibliotecas.


In [1]:
!pip install pandas
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 193.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 196.0 MB/s eta 0:00:00


## Importar bibliotecas y definir funciones auxiliares


Importaremos las siguientes bibliotecas.


In [2]:
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
#NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np

### Análisis de Datos 


Cargar el conjunto de datos de Space X, de la sección anterior.


In [4]:
df=pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


Identificar y calcular el porcentaje de valores faltantes en cada atributo.


In [5]:
df.isnull().sum()/len(df)*100

FlightNumber       0.000000
Date               0.000000
BoosterVersion     0.000000
PayloadMass        0.000000
Orbit              0.000000
LaunchSite         0.000000
Outcome            0.000000
Flights            0.000000
GridFins           0.000000
Reused             0.000000
Legs               0.000000
LandingPad        28.888889
Block              0.000000
ReusedCount        0.000000
Serial             0.000000
Longitude          0.000000
Latitude           0.000000
dtype: float64

Identifique qué columnas son numéricas y cuáles categóricas:


In [6]:
df.dtypes

FlightNumber        int64
Date                  str
BoosterVersion        str
PayloadMass       float64
Orbit                 str
LaunchSite            str
Outcome               str
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad            str
Block             float64
ReusedCount         int64
Serial                str
Longitude         float64
Latitude          float64
dtype: object

### TAREA 1: Calcular el número de lanzamientos en cada sitio

Los datos contienen varias instalaciones de lanzamiento de SpaceX: <a href='https://en.wikipedia.org/wiki/List_of_Cape_Canaveral_and_Merritt_Island_launch_sites'>Complejo de Lanzamiento Espacial de Cabo Cañaveral 40 <b>VAFB SLC 4E</b>, Complejo de Lanzamiento Espacial de la Base de la Fuerza Aérea Vandenberg 4E <b>(SLC-4E)</b>, Complejo de Lanzamiento del Centro Espacial Kennedy 39A <b>KSC LC 39A</b>. La ubicación de cada lanzamiento se indica en la columna <code>LaunchSite</code>.


A continuación, veamos el número de lanzamientos para cada sitio.

Utiliza el método `value_counts()` en la columna `LaunchSite` para determinar el número de lanzamientos en cada sitio:

In [9]:
# Aplicar value_counts() en la columna LaunchSite

númerosLanzamiento = df['LaunchSite'].value_counts()
print(númerosLanzamiento)

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64


Cada lanzamiento tiene como objetivo una órbita específica, y estos son algunos tipos de órbita comunes:



* <b>LEO</b>: La órbita terrestre baja (LEO) es una órbita centrada en la Tierra con una altitud de 2000 km (1200 millas) o menos (aproximadamente un tercio del radio de la Tierra),[1] o con al menos 11,25 periodos por día (un periodo orbital de 128 minutos o menos) y una excentricidad menor de 0,25.[2] La mayoría de los objetos artificiales en el espacio exterior se encuentran en LEO <a href='https://en.wikipedia.org/wiki/Low_Earth_orbit'>[1]</a>.

* <b>VLEO</b>: Las órbitas terrestres muy bajas (VLEO) se definen como órbitas con una altitud media inferior a 450 km. Operar en estas órbitas puede proporcionar una serie de beneficios a las naves espaciales de observación de la Tierra, ya que la nave opera más cerca del objeto de observación<a href='https://www.researchgate.net/publication/271499606_Very_Low_Earth_Orbit_mission_concepts_for_Earth_Observation_Benefits_and_challenges'>[2]</a>.

* <b>Órbita de Transferencia Geoestacionaria</b>: Una órbita de transferencia geoestacionaria es una órbita terrestre elíptica que se utiliza para transferir satélites de la órbita terrestre baja (LEO) a la órbita geoestacionaria (GEO). En una GTO, el perigeo (punto más cercano a la Tierra) es mucho menor que la altitud de la GEO, mientras que el apogeo (punto más lejano) alcanza aproximadamente 35 786 kilómetros (22 236 millas) sobre el ecuador terrestre, la altitud de una órbita geoestacionaria. Los satélites en órbita de transferencia geoestacionaria (GTO) utilizan propulsión a bordo para circularizar su órbita a la altitud geoestacionaria (GEO), donde pueden proporcionar servicios como monitoreo meteorológico, comunicaciones y vigilancia. <a href="https://www.space.com/29222-geosynchronous-orbit.html" >[3] </a>.

* <b>Órbita sincrónica con el Sol (SSO)</b>: Es una órbita sincrónica con el Sol, también llamada órbita heliosincrónica. Es una órbita casi polar alrededor de un planeta, en la que el satélite pasa sobre cualquier punto de la superficie del planeta a la misma hora solar media local. <a href="https://en.wikipedia.org/wiki/Sun-synchronous_orbit">[4] <a>.

* <b>Órbita ES-L1</b>: En los puntos de Lagrange, las fuerzas gravitatorias de los dos cuerpos grandes se anulan de tal manera que un objeto pequeño colocado en órbita allí se encuentra en equilibrio con respecto al centro de masa de los cuerpos grandes. L1 es uno de esos puntos entre el Sol y la Tierra <a href="https://en.wikipedia.org/wiki/Lagrange_point#L1_point">[5]</a>.

* <b>Órbita altamente elíptica</b> (HEO): una órbita elíptica con alta excentricidad, generalmente referida a una órbita alrededor de la Tierra <a href="https://en.wikipedia.org/wiki/Highly_elliptical_orbit">[6]</a>.

* <b>ISS</b>: una estación espacial modular (satélite artificial habitable) en órbita terrestre baja. Es un proyecto de colaboración multinacional entre cinco agencias espaciales participantes: NASA (Estados Unidos), Roscosmos (Rusia), JAXA (Japón), ESA (Europa) y CSA (Canadá)<a href="https://en.wikipedia.org/wiki/International_Space_Station"> [7] </a>

* <b> MEO </b> Órbitas geocéntricas con altitudes que van desde los 2000 km (1200 mi) hasta justo por debajo de la órbita geosíncrona, a 35 786 kilómetros (22 236 mi). También conocidas como órbitas circulares intermedias. Estas se encuentran generalmente a 20 200 kilómetros (12 600 millas) o 20 650 kilómetros (12 830 millas), con un período orbital de 12 horas <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [8] </a>

* <b> HEO </b> Órbitas geocéntricas por encima de la altitud de la órbita geosíncrona (35 786 km o 22 236 millas) <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [9] </a>

* <b> GEO </b> Es una órbita geosíncrona circular a 35 786 kilómetros (22 236 millas) sobre el ecuador terrestre, siguiendo la dirección de rotación de la Tierra <a href="https://en.wikipedia.org/wiki/Geostationary_orbit"> [10] </a>

* <b> PO </b> Es una Tipo de satélites en los que un satélite pasa por encima o casi por encima de ambos polos del cuerpo que orbita (generalmente un planeta como la Tierra <a href="https://en.wikipedia.org/wiki/Polar_orbit"> [11] </a>

Algunos se muestran en el siguiente gráfico:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/Orbits.png)


### TAREA 2: Calcular el número y la frecuencia de cada órbita


Utilice el método `.value_counts()` para determinar el número y la frecuencia de cada órbita en la columna `Orbit`.

Nota: No cuente las órbitas GTO, ya que son órbitas de transferencia y no geoestacionarias.


In [11]:
# Aplicar value_counts a la columna Orbit

númeroFrecuenciaOrbitas = df[df['Orbit'] != 'GTO']['Orbit'].value_counts()

print(númeroFrecuenciaOrbitas)


Orbit
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64


### TAREA 3: Calcular el número y la frecuencia de los resultados de la misión de las órbitas


Utilice el método <code>.value_counts()</code> en la columna <code>Outcome</code> para determinar el número de <code>landing_outcomes</code>. Luego asígnelo a una variable landing_outcomes.


In [12]:
# landing_outcomes = valores en la columna Outcome
landing_outcomes = df['Outcome'].value_counts()

print(landing_outcomes)

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64


<code>True Ocean</code> significa que el resultado de la misión fue aterrizado con éxito en una región específica del océano, mientras que <code>False Ocean</code> significa que el resultado de la misión no fue aterrizado con éxito en una región específica del océano. <code>True RTLS</code> significa que el resultado de la misión fue aterrizado con éxito en una plataforma terrestre. <code>False RTLS</code> significa que el resultado de la misión no fue aterrizado con éxito en una plataforma terrestre. <code>True ASDS</code> significa que el resultado de la misión fue aterrizado con éxito en un barco no tripulado. <code>False ASDS</code> significa que el resultado de la misión no fue aterrizado con éxito en un barco no tripulado. <code>None ASDS</code> y <code>None None</code> representan un fallo en el aterrizaje.


In [13]:
for i,outcome in enumerate(landing_outcomes.keys()):
    print(i,outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


Creamos un conjunto de resultados en los que la segunda etapa no tuvo éxito:


In [14]:
bad_outcomes=set(landing_outcomes.keys()[[1,3,5,6,7]])
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

### TAREA 4: Crear una etiqueta de resultado de aterrizaje a partir de la columna Resultado


Utilizando <code>Outcome</code>, crea una lista donde el elemento sea cero si la fila correspondiente en <code>Outcome</code> está en el conjunto <code>bad_outcome</code>; de lo contrario, será uno. Luego asígnalo a la variable <code>landing_class</code>:


In [16]:
# landing_class = 0 if bad_outcome    landing_class = 0 si el resultado es malo
# landing_class = 1 otherwise   landing_class = 1 en caso contrario

landing_class = [0 if x in bad_outcomes else 1 for x in df['Outcome']]
df['Class'] = landing_class
print(df[['Outcome', 'Class']].head(10))

       Outcome  Class
0    None None      0
1    None None      0
2    None None      0
3  False Ocean      0
4    None None      0
5    None None      0
6   True Ocean      1
7   True Ocean      1
8    None None      0
9    None None      0


Esta variable representará la variable de clasificación que representa el resultado de cada lanzamiento. Si el valor es cero, la primera etapa no aterrizó correctamente; uno significa que la primera etapa aterrizó correctamente.



In [17]:
df['Class']=landing_class
df[['Class']].head(8)

,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,1
7,1


In [18]:
df.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


Podemos utilizar la siguiente línea de código para determinar la tasa de éxito:


In [19]:
df["Class"].mean()

np.float64(0.6666666666666666)

Ahora podemos exportarlo a un archivo CSV para la siguiente sección, pero para que las respuestas sean coherentes, en el próximo laboratorio proporcionaremos datos en un rango de fechas preseleccionado.


<code>df.to_csv("dataset_part_2.csv", index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a> is a Data Scientist at IBM and pursuing a Master of Management in Artificial intelligence degree at Queen's University.


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-08-31        | 1.1     | Lakshmi Holla    | Changed Markdown |
| 2020-09-20        | 1.0     | Joseph     | Modified Multiple Areas |
| 2020-11-04        | 1.1.    | Nayef      | updating the input data |
| 2021-05-026       | 1.1.    | Joseph      | updating the input data |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
